# Fink/LSST — Dipole Concentration per diaObject

## Strategy

The **conesearch** endpoint returns **one alert per diaObject** (the most recent detection).
It is therefore impossible to count dipoles directly from the conesearch.
The conesearch alert does, however, carry `r:nDiaSources`, which gives the total number
of detections accumulated for that object since the start of the survey.

The correct pipeline is:

1. **Cone-search** all DDFs (same approach as `01_fink_block_flatlightcurves.ipynb`).
2. **Pre-select** objects with `r:nDiaSources >= NDIASOURCES_MIN` (e.g. 1000) to keep
   only well-sampled objects where a dipole rate is statistically meaningful.
3. **Download the full diaSources** for the pre-selected objects via `/api/v1/sources`,
   requesting all dipole columns (`r:isDipole`, `r:dipoleAngle`, `r:dipoleLength`, …).
4. **Count dipoles** per object and per band from the diaSources.
5. **Plot light curves** for the objects with the highest dipole counts, with the
   standard three-panel layout (light curve + nightly dipole histogram + dipole morphology).

## Scientific goals

- Show that dipoles are **not uniformly distributed** across well-observed diaObjects.
- Identify which objects concentrate the dipoles and in which bands (g, r dominant?).
- Determine whether dipoles are **clustered in specific nights** or recurrent across the survey.
- Check whether the **dipole angle / length is stable** (systematic offset) or random (noise).


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-26
- last update : 2026-05-26

## 1. Imports & configuration

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("ipympl not found → %matplotlib inline")

In [ ]:
# ── Fink API ──────────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org"

# ── DDF cone-search parameters ────────────────────────────────────────────────
CONE_RADIUS = 1800.0  # arcsec (0.5 deg per DDF, same as notebook 01)
# CONE_RADIUS   = 3600.0   # arcsec (1.0 deg per DDF, same as notebook 01)
CONE_N_ALERTS = 2000  # max alerts per cone-search call

# ── Pre-selection threshold on nDiaSources (from conesearch) ─────────────────
# Only objects with >= NDIASOURCES_MIN total detections are selected.
# At this level a dipole rate of a few percent is already statistically meaningful.
# Recommended: 1000.  Reduce to 500 or 200 if too few objects survive.
NDIASOURCES_MIN = 500

# ── Light curve plotting ──────────────────────────────────────────────────────
TOP_N_OBJECTS = 10  # max number of objects shown in detailed light curve plots

# ── DDFs (must match notebook 01c) ───────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "DIPOLES_03"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Output data: {os.path.abspath(DIR_DATA)}")
print(f"Figures    : {os.path.abspath(DIR_FIGS)}")

# ── Plotting style ────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Configuration done.  NDIASOURCES_MIN={NDIASOURCES_MIN}")

## 2. API wrappers

Minimal wrappers, consistent with `01_fink_block_flatlightcurves.ipynb`.

In [ ]:
def _post_json(url: str, payload: dict, timeout: int = 90):
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()


def fetch_conesearch(
    ra: float, dec: float, radius: float, n: int = 2000, columns: str | None = None
) -> pd.DataFrame:
    """
    Cone-search via /api/v1/conesearch.  IMPORTANT: use 'r:' column prefix.
    Returns ONE alert per recent detection (not one per object).
    """
    payload = {"ra": ra, "dec": dec, "radius": radius, "n": n, "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/conesearch", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_sources(diaObjectId, columns: str | None = None) -> pd.DataFrame:
    """Fetch ALL diaSources for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/sources", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def parse_dipole_bool(series: pd.Series) -> pd.Series:
    """Convert a dipole boolean column (bool / int / str) to a proper bool Series."""

    def _to_bool(v):
        if isinstance(v, bool):
            return v
        if isinstance(v, (int, float)):
            return bool(v)
        if isinstance(v, str):
            return v.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_to_bool)


print("API wrappers defined.")

## 3. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """Convert MJD (TAI) → list of 'YYYY-MM-DD' strings."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 8) -> None:
    """Add a secondary x-axis on top of *ax* showing calendar dates (YYYY-MM-DD)."""
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return
    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return
    tick_mjd = np.linspace(mjd_lo, mjd_hi, max(3, min(n_ticks, len(finite))))
    tick_lbls = mjd_to_datestr(tick_mjd)
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=7, labelpad=6)


print("Utility functions defined.")

## 4. Cone-search all DDFs — build the pre-selection catalogue

For each DDF we retrieve the latest alert per detection, deduplicate by `diaObjectId`,
and keep only objects with `r:nDiaSources >= NDIASOURCES_MIN`.

Columns requested are minimal: `r:diaObjectId`, `r:nDiaSources`, `r:ra`, `r:dec`,
plus a few crossmatch columns for later identification.

In [ ]:
COLS_CONE = (
    "r:diaObjectId,r:nDiaSources,r:ra,r:dec,r:band,"
    "f:xm_gaiadr3_DR3Name,f:xm_gaiadr3_VarFlag,"
    "f:xm_simbad_otype,"
    "f:xm_vsx_Type,f:xm_tns_fullname,"
    "f:main_label_crossmatch"
)

NULL_VALS = {"", "None", "nan", "Fail", "null", "NaN"}


def _mode_non_null(series: pd.Series):
    """Return the mode of a Series after dropping null-like values."""
    vals = series.dropna().astype(str)
    vals = vals[~vals.isin(NULL_VALS)]
    return vals.mode().iloc[0] if not vals.empty else None


# Crossmatch columns that may vary across alerts of the same object
XM_COLS = [
    "f:xm_gaiadr3_DR3Name",
    "f:xm_gaiadr3_VarFlag",
    "f:xm_simbad_otype",
    "f:xm_vsx_Type",
    "f:xm_tns_fullname",
    "f:main_label_crossmatch",
]

# Dict: diaObjectId (str) → metadata
presel: dict[str, dict] = {}

for field_name, (ra, dec) in DEEP_FIELDS.items():
    print(f'\n── {field_name}  RA={ra:.4f}  Dec={dec:.4f}  r={CONE_RADIUS:.0f}"')
    try:
        df_cone = fetch_conesearch(ra, dec, CONE_RADIUS, n=CONE_N_ALERTS, columns=COLS_CONE)
    except Exception as e:
        print(f"  ERROR: {e}")
        continue

    if df_cone.empty:
        print("  No alerts.")
        continue
    print(f"  Alerts returned: {len(df_cone)}")

    oid_col = "r:diaObjectId"
    nsrc_col = "r:nDiaSources"
    if oid_col not in df_cone.columns:
        print(f"  Column {oid_col!r} absent — skipping.")
        continue

    df_cone[nsrc_col] = pd.to_numeric(df_cone.get(nsrc_col, 0), errors="coerce").fillna(0)

    # Aggregate crossmatch columns (mode of non-null values across all alerts
    # of the same object — fix for the alert-level inconsistency described in 01)
    agg_rows = []
    for oid, grp in df_cone.groupby(oid_col):
        row = {
            oid_col: oid,
            nsrc_col: int(grp[nsrc_col].max()),
            "r:ra": float(grp["r:ra"].iloc[0]),
            "r:dec": float(grp["r:dec"].iloc[0]),
        }
        for col in XM_COLS:
            row[col] = _mode_non_null(grp[col]) if col in grp.columns else None
        agg_rows.append(row)

    df_obj = pd.DataFrame(agg_rows)
    print(f"  Unique objects: {len(df_obj)}")

    df_ok = df_obj[df_obj[nsrc_col] >= NDIASOURCES_MIN]
    print(f"  With nDiaSources>={NDIASOURCES_MIN}: {len(df_ok)}")

    for _, row in df_ok.iterrows():
        oid = str(row[oid_col])
        if oid not in presel:
            presel[oid] = {
                "nDiaSources": int(row[nsrc_col]),
                "ra": row["r:ra"],
                "dec": row["r:dec"],
                "field": field_name,
                "gaia_name": row.get("f:xm_gaiadr3_DR3Name"),
                "gaia_var": row.get("f:xm_gaiadr3_VarFlag"),
                "simbad": row.get("f:xm_simbad_otype"),
                "vsx": row.get("f:xm_vsx_Type"),
                "tns": row.get("f:xm_tns_fullname"),
                "label": row.get("f:main_label_crossmatch"),
            }
    time.sleep(0.5)

print(f"\n=== Pre-selected objects (nDiaSources>={NDIASOURCES_MIN}): {len(presel)} ===")
by_field = pd.Series({oid: v["field"] for oid, v in presel.items()}).value_counts()
print(by_field.to_string())

In [ ]:
# ── Build and display the pre-selection catalogue ─────────────────────────────
df_presel = (
    pd.DataFrame([{"diaObjectId": oid, **meta} for oid, meta in presel.items()])
    .sort_values("nDiaSources", ascending=False)
    .reset_index(drop=True)
)

print(f"Pre-selection catalogue: {len(df_presel)} objects")
display(df_presel.head(20))

# Distribution of nDiaSources
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
nv = df_presel["nDiaSources"].values
axes[0].hist(nv, bins=40, color="steelblue", edgecolor="white", lw=0.3)
axes[0].set_xlabel("nDiaSources")
axes[0].set_ylabel("N objects")
axes[0].set_title(f"nDiaSources distribution (all >= {NDIASOURCES_MIN})")
axes[1].hist(
    nv,
    bins=np.logspace(np.log10(NDIASOURCES_MIN), np.log10(nv.max() + 1), 30),
    color="steelblue",
    edgecolor="white",
    lw=0.3,
)
axes[1].set_xscale("log")
axes[1].set_xlabel("nDiaSources")
axes[1].set_title("nDiaSources distribution (log)")
plt.tight_layout()
savefig(f"nDiaSources_distribution_min{NDIASOURCES_MIN}")
plt.show()

# Save catalogue
df_presel.to_parquet(os.path.join(DIR_DATA, "presel_catalogue.parquet"), index=False)
df_presel.to_csv(os.path.join(DIR_DATA, "presel_catalogue.csv"), index=False)
print("Pre-selection catalogue saved.")

## 5. Download full diaSources for all pre-selected objects

All dipole columns are requested explicitly:
`r:isDipole`, `r:dipoleAngle`, `r:dipoleLength`, `r:dipoleChi2`,
`r:dipoleFluxDiff`, `r:dipoleMeanFlux`, `r:dipoleNdata`, …

This step can take a few minutes depending on the number of selected objects.

In [ ]:
COLS_SRC = (
    "r:diaObjectId,r:diaSourceId,r:midpointMjdTai,"
    "r:psfFlux,r:psfFluxErr,r:scienceFlux,r:scienceFluxErr,"
    "r:templateFlux,r:templateFluxErr,"
    "r:apFlux,r:apFluxErr,"
    "r:band,r:ra,r:dec,r:snr,"
    "r:visit,r:detector,r:x,r:y,"
    "r:isDipole,r:isNegative,r:dipoleFitAttempted,"
    "r:dipoleFluxDiff,r:dipoleFluxDiffErr,"
    "r:dipoleMeanFlux,r:dipoleMeanFluxErr,"
    "r:dipoleLength,r:dipoleAngle,"
    "r:dipoleNdata,r:dipoleChi2"
)


def _cast_src(df: pd.DataFrame) -> pd.DataFrame:
    """Cast columns to appropriate types after API download."""
    for col in ("r:isDipole", "r:isNegative", "r:dipoleFitAttempted"):
        if col in df.columns:
            df[col] = parse_dipole_bool(df[col].fillna(False))
    for col in (
        "r:midpointMjdTai",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:scienceFlux",
        "r:scienceFluxErr",
        "r:templateFlux",
        "r:templateFluxErr",
        "r:apFlux",
        "r:apFluxErr",
        "r:snr",
        "r:dipoleFluxDiff",
        "r:dipoleFluxDiffErr",
        "r:dipoleMeanFlux",
        "r:dipoleMeanFluxErr",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleNdata",
        "r:dipoleChi2",
        "r:x",
        "r:y",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in ("r:visit", "r:detector"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    return df


# Download — sorted by nDiaSources descending so the richest objects come first
oids_sorted = df_presel["diaObjectId"].tolist()
src_cache: dict[str, pd.DataFrame] = {}  # oid → raw diaSources DataFrame

print(f"Downloading diaSources for {len(oids_sorted)} objects…")
for i, oid in enumerate(oids_sorted):
    try:
        df = fetch_sources(oid, columns=COLS_SRC)
        df = _cast_src(df)
        src_cache[oid] = df
        n_dip = int(df["r:isDipole"].sum()) if ("r:isDipole" in df.columns and not df.empty) else 0
        print(
            f"  [{i + 1:3d}/{len(oids_sorted)}] {oid}  "
            f"n_src={len(df):5d}  n_dipoles={n_dip:4d}  "
            f"field={presel[oid]['field']}"
        )
    except Exception as e:
        print(f"  [{i + 1:3d}/{len(oids_sorted)}] {oid}  ERROR: {e}")
        src_cache[oid] = pd.DataFrame()
    time.sleep(0.2)

print(f"\nDownload complete: {sum(1 for df in src_cache.values() if not df.empty)} objects with data.")

## 6. Compute per-object dipole statistics from diaSources

Now that we have the complete diaSources, we can count dipoles properly:
- `n_src`          : total number of diaSources downloaded
- `n_dipoles`      : number of diaSources with `r:isDipole == True`
- `dipole_fraction`: `n_dipoles / n_src`
- `n_dipoles_<b>`  : per-band dipole counts (b ∈ ugrizy)

In [ ]:
stat_rows = []
for oid, df in src_cache.items():
    if df.empty or "r:isDipole" not in df.columns:
        continue
    meta = presel[oid]
    n_src = len(df)
    n_dip = int(df["r:isDipole"].sum())
    row = {
        "diaObjectId": oid,
        "field": meta["field"],
        "nDiaSources": meta["nDiaSources"],  # from conesearch (max ever)
        "n_src": n_src,  # actually downloaded
        "n_dipoles": n_dip,
        "dipole_fraction": n_dip / n_src if n_src > 0 else np.nan,
        "ra": meta["ra"],
        "dec": meta["dec"],
        "gaia_name": meta.get("gaia_name"),
        "simbad": meta.get("simbad"),
        "label": meta.get("label"),
    }
    # Per-band dipole counts
    if "r:band" in df.columns:
        band_dip = df[df["r:isDipole"]].groupby("r:band").size().reindex(BAND_ORDER, fill_value=0)
        for b in BAND_ORDER:
            row[f"n_dip_{b}"] = int(band_dip.get(b, 0))
    stat_rows.append(row)

df_stats = pd.DataFrame(stat_rows).sort_values("n_dipoles", ascending=False).reset_index(drop=True)

print(f"Statistics computed for {len(df_stats)} objects.")
print(f"  with >= 1 dipole  : {(df_stats['n_dipoles'] >= 1).sum()}")
print(f"  with >= 5 dipoles : {(df_stats['n_dipoles'] >= 5).sum()}")
print(f"  with >= 20 dipoles: {(df_stats['n_dipoles'] >= 20).sum()}")
display(df_stats.head(20))

df_stats.to_parquet(os.path.join(DIR_DATA, "dipole_stats_from_sources.parquet"), index=False)
df_stats.to_csv(os.path.join(DIR_DATA, "dipole_stats_from_sources.csv"), index=False)
print("Saved dipole statistics.")

## 7. Stacked histogram: dipole count per object by band

Each bar = one `diaObjectId` (pre-selected with `nDiaSources >= NDIASOURCES_MIN`).
The stacked colours show the per-band dipole count.
Objects are sorted by total dipole count descending.

In [ ]:
dip_cols = [f"n_dip_{b}" for b in BAND_ORDER if f"n_dip_{b}" in df_stats.columns]
df_dip_nonzero = df_stats[df_stats["n_dipoles"] > 0].copy()

if df_dip_nonzero.empty:
    print("No dipoles found in the pre-selected objects.")
else:
    N_SHOW = min(80, len(df_dip_nonzero))
    top_df = df_dip_nonzero.head(N_SHOW)
    x_pos = np.arange(len(top_df))
    bottom = np.zeros(len(top_df))

    fig, ax = plt.subplots(figsize=(max(12, N_SHOW * 0.22), 5))
    for band in BAND_ORDER:
        col = f"n_dip_{band}"
        if col not in top_df.columns:
            continue
        vals = top_df[col].values.astype(float)
        ax.bar(
            x_pos,
            vals,
            bottom=bottom,
            color=BAND_COLORS[band],
            edgecolor="white",
            lw=0.3,
            label=f"band {band}",
            width=0.85,
        )
        bottom += vals

    ax.set_xticks(x_pos)
    ax.set_xticklabels(
        # [str(oid)[-8:] for oid in top_df["diaObjectId"]],
        [str(oid) for oid in top_df["diaObjectId"]],
        rotation=90,
        fontsize=6,
    )
    ax.set_xlabel("diaObjectId (last 8 digits)")
    ax.set_ylabel("Number of dipole detections")
    ax.set_title(
        f"Dipole count per diaObject — top {N_SHOW} (stacked by band)\n"
        f"Pre-selection: nDiaSources >= {NDIASOURCES_MIN}  |  "
        f"Objects with >=1 dipole: {len(df_dip_nonzero)}"
    )
    ax.legend(loc="upper right", fontsize=8, ncol=3)
    plt.tight_layout()
    savefig(f"dipole_stacked_per_object_min{NDIASOURCES_MIN}")
    plt.show()

In [ ]:
# ── Distribution of dipole counts (log-log) and Lorenz curve ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
counts = df_dip_nonzero["n_dipoles"].values

# Linear histogram
axes[0].hist(counts, bins=40, color="steelblue", edgecolor="white", lw=0.3)
axes[0].set_xlabel("n_dipoles per object")
axes[0].set_ylabel("N objects")
axes[0].set_title(f"Dipole count distribution (nDiaSrc>={NDIASOURCES_MIN})")

# Log-log histogram
bins_log = np.logspace(0, np.log10(counts.max() + 1), 30)
axes[1].hist(counts, bins=bins_log, color="tomato", edgecolor="white", lw=0.3)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("n_dipoles per object")
axes[1].set_title("Dipole count distribution (log-log)")

# Lorenz curve
sc = np.sort(counts)[::-1]
cum = np.cumsum(sc) / sc.sum()
obj = np.arange(1, len(sc) + 1) / len(sc)
axes[2].plot(obj * 100, cum * 100, color="steelblue", lw=2)
axes[2].plot([0, 100], [0, 100], "--", color="grey", lw=1, label="equal distribution")
idx10 = max(1, int(0.10 * len(sc)))
frac10 = cum[idx10 - 1] * 100
axes[2].axvline(10, color="tomato", lw=1, ls=":")
axes[2].axhline(frac10, color="tomato", lw=1, ls=":")
axes[2].text(11, frac10 + 1, f"top 10% objects\n→ {frac10:.0f}% of dipoles", color="tomato", fontsize=8)
axes[2].set_xlabel("Fraction of objects (%)")
axes[2].set_ylabel("Cumulative dipoles (%)")
axes[2].set_title("Lorenz curve — dipole concentration")
axes[2].legend(fontsize=8)

plt.tight_layout()
savefig(f"dipole_distribution_lorenz_min{NDIASOURCES_MIN}")
plt.show()

## 8. Select top high-dipole objects for light curve inspection

Sort by `n_dipoles` descending and keep the top `TOP_N_OBJECTS`.

In [ ]:
top_sel = df_stats[df_stats["n_dipoles"] > 0].head(TOP_N_OBJECTS).copy()
print(f"Top {TOP_N_OBJECTS} objects by dipole count:")
display(
    top_sel[
        [
            "diaObjectId",
            "field",
            "nDiaSources",
            "n_src",
            "n_dipoles",
            "dipole_fraction",
            "gaia_name",
            "simbad",
            "label",
        ]
    ]
)

# 2-D scatter: n_src vs n_dipoles
fig, ax = plt.subplots(figsize=(7, 5))
for fld in DEEP_FIELDS:
    sub = df_stats[df_stats["field"] == fld]
    ax.scatter(sub["n_src"], sub["n_dipoles"], s=12, alpha=0.5, label=fld)
ax.scatter(
    top_sel["n_src"].values,
    top_sel["n_dipoles"].values,
    s=90,
    marker="*",
    color="gold",
    edgecolors="k",
    lw=0.5,
    zorder=5,
    label=f"top {TOP_N_OBJECTS}",
)
ax.set_xlabel("n_src (diaSources downloaded)")
ax.set_ylabel("n_dipoles")
ax.set_title(f"Sources vs dipoles per diaObject  [nDiaSrc>={NDIASOURCES_MIN}]")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
savefig(f"nsrc_vs_ndipoles_scatter_min{NDIASOURCES_MIN}")
plt.show()

## 9. Light curve plots for top high-dipole objects

Three-panel figure per object:

**Panel 1 — psfFlux light curve**
- One colour per band (`BAND_COLORS`).
- Dipole detections (`r:isDipole == True`) circled with a grey open marker.
- Primary x-axis: MJD.  Secondary x-axis: date (YYYY-MM-DD).

**Panel 2 — nightly dipole histogram**
- Stacked bars per band, one bar per night (floor of MJD).
- Secondary y-axis: cumulative dipole count.

**Panel 3 — dipole morphology**
- `dipoleLength` (arcsec) vs MJD — left y-axis.
- `dipoleAngle` (degrees mod 360) vs MJD — right y-axis.
- Stable angle → systematic template offset; random → noise artefact.

In [ ]:
def plot_object_lc(
    oid: str,
    df_src: pd.DataFrame,
    meta: dict,
    stat: dict,
    flux_col: str = "r:psfFlux",
    ferr_col: str = "r:psfFluxErr",
) -> None:
    """
    Three-panel light curve + dipole histogram + morphology for one diaObject.

    Parameters
    ----------
    oid     : diaObjectId string
    df_src  : full diaSources DataFrame for this object (from src_cache)
    meta    : metadata dict from `presel`
    stat    : statistics row from `df_stats` (as dict)
    """
    if df_src.empty:
        print(f"  {oid}: empty diaSources — skipping.")
        return

    df = df_src.sort_values("r:midpointMjdTai").copy()
    df["is_dipole"] = df["r:isDipole"].fillna(False).astype(bool) if "r:isDipole" in df.columns else False
    mjd_all = pd.to_numeric(df["r:midpointMjdTai"], errors="coerce").values

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(13, 10),
        gridspec_kw={"height_ratios": [3, 1.5, 1.5]},
    )

    # ── Panel 1 : psfFlux light curve ────────────────────────────────────────
    ax1 = axes[0]
    for band in BAND_ORDER:
        sub = df[df["r:band"] == band] if "r:band" in df.columns else pd.DataFrame()
        if sub.empty:
            continue
        flux = pd.to_numeric(sub[flux_col], errors="coerce").values
        ferr = pd.to_numeric(sub[ferr_col], errors="coerce").values if ferr_col in sub.columns else None
        mjd = pd.to_numeric(sub["r:midpointMjdTai"], errors="coerce").values
        color = BAND_COLORS[band]

        ax1.errorbar(
            mjd,
            flux,
            yerr=ferr,
            fmt="o",
            ms=4,
            lw=0.8,
            capsize=2,
            capthick=0.8,
            color=color,
            ecolor=color,
            alpha=0.8,
            label=f"{band} (n={len(sub)})",
        )

        # Grey ring around dipole points
        dip = sub[sub["is_dipole"]]
        if not dip.empty:
            flux_d = pd.to_numeric(dip[flux_col], errors="coerce").values
            mjd_d = pd.to_numeric(dip["r:midpointMjdTai"], errors="coerce").values
            ax1.scatter(
                mjd_d,
                flux_d,
                s=130,
                facecolors="none",
                edgecolors="grey",
                linewidths=1.8,
                zorder=5,
                label=f"dipole {band} (n={len(dip)})",
            )

    ax1.axhline(0, color="k", lw=0.5, ls="--", alpha=0.4)
    ax1.set_ylabel(f"{flux_col.split(':')[1]} (nJy)")
    ax1.legend(loc="best", fontsize=7, ncol=4)
    title = (
        f"diaObjectId={oid}  field={meta['field']}  "
        f"nDiaSrc={meta['nDiaSources']}  n_downloaded={stat['n_src']}  "
        f"n_dip={stat['n_dipoles']}  frac={stat['dipole_fraction'] * 100:.1f}%"
    )
    if meta.get("gaia_name") and str(meta["gaia_name"]) not in ("nan", "None", ""):
        title += f"  Gaia={meta['gaia_name']}"
    if meta.get("simbad") and str(meta["simbad"]) not in ("nan", "None", ""):
        title += f"  SIMBAD={meta['simbad']}"
    ax1.set_title(title, fontsize=8)
    add_date_axis_on_top(ax1, mjd_all, n_ticks=8)

    # ── Panel 2 : nightly dipole histogram ───────────────────────────────────
    ax2 = axes[1]
    df_dip = df[df["is_dipole"]].copy()

    if not df_dip.empty and "r:band" in df_dip.columns:
        df_dip["night"] = np.floor(pd.to_numeric(df_dip["r:midpointMjdTai"], errors="coerce").values).astype(
            int
        )
        night_band = (
            df_dip.groupby(["night", "r:band"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=BAND_ORDER, fill_value=0)
        )
        night_band["total"] = night_band.sum(axis=1)
        nights_mjd = night_band.index.values.astype(float) + 0.5

        bottom = np.zeros(len(night_band))
        for band in BAND_ORDER:
            if band not in night_band.columns:
                continue
            vals = night_band[band].values.astype(float)
            ax2.bar(
                nights_mjd,
                vals,
                bottom=bottom,
                width=0.8,
                color=BAND_COLORS[band],
                edgecolor="white",
                lw=0.3,
                label=f"band {band}",
            )
            bottom += vals

        cum = np.cumsum(night_band["total"].values)
        ax2r = ax2.twinx()
        ax2r.step(nights_mjd, cum, where="post", color="k", lw=1.5, ls="--", alpha=0.6, label="cumulative")
        ax2r.set_ylabel("Cumulative dipoles", fontsize=8)
        ax2r.tick_params(axis="y", labelsize=8)

    ax2.set_ylabel("N dipoles per night")
    ax2.set_xlabel("MJD (TAI)")
    ax2.legend(loc="upper left", fontsize=7, ncol=3)

    if np.isfinite(mjd_all).sum() > 1:
        xlim = (mjd_all[np.isfinite(mjd_all)].min() - 1, mjd_all[np.isfinite(mjd_all)].max() + 1)
        ax1.set_xlim(xlim)
        ax2.set_xlim(xlim)

    # ── Panel 3 : dipole morphology ───────────────────────────────────────────
    ax3 = axes[2]
    if not df_dip.empty:
        for band in BAND_ORDER:
            sub = df_dip[df_dip["r:band"] == band] if "r:band" in df_dip.columns else pd.DataFrame()
            if sub.empty or "r:dipoleLength" not in sub.columns:
                continue
            dl = pd.to_numeric(sub["r:dipoleLength"], errors="coerce").values
            mjd_b = pd.to_numeric(sub["r:midpointMjdTai"], errors="coerce").values
            ax3.scatter(mjd_b, dl, s=25, color=BAND_COLORS[band], marker="o", label=f"length {band}")

        if "r:dipoleAngle" in df_dip.columns:
            ax3r = ax3.twinx()
            for band in BAND_ORDER:
                sub = df_dip[df_dip["r:band"] == band] if "r:band" in df_dip.columns else pd.DataFrame()
                if sub.empty:
                    continue
                da = pd.to_numeric(sub["r:dipoleAngle"], errors="coerce").values % 360
                mjd_b = pd.to_numeric(sub["r:midpointMjdTai"], errors="coerce").values
                ax3r.scatter(mjd_b, da, s=25, color=BAND_COLORS[band], marker="^", alpha=0.6)
            ax3r.set_ylabel("Dipole angle (deg)", fontsize=8, color="grey")
            ax3r.set_ylim(0, 360)
            ax3r.tick_params(axis="y", labelcolor="grey", labelsize=8)

        ax3.set_ylabel("Dipole length (arcsec)")
        ax3.set_xlabel("MJD (TAI)")
        ax3.legend(loc="upper left", fontsize=7, ncol=3)
        ax3.set_xlim(ax1.get_xlim())

    plt.tight_layout()
    savefig(f"lc_{oid}")
    plt.show()


print("plot_object_lc() defined.")

In [ ]:
# ── Plot all top objects ──────────────────────────────────────────────────────
for _, srow in top_sel.iterrows():
    oid = str(srow["diaObjectId"])
    print(f"\n=== {oid}  field={srow['field']}  nDiaSrc={srow['nDiaSources']}  n_dip={srow['n_dipoles']} ===")
    plot_object_lc(
        oid=oid,
        df_src=src_cache.get(oid, pd.DataFrame()),
        meta=presel[oid],
        stat=srow.to_dict(),
    )

print("Done.")

## 10. Angular correlation of dipole angles per object

For each selected object, compute the **angular dispersion** of the dipole position
angle across all dipole detections:

$$\sigma_{\theta} = \frac{1}{2}\sqrt{-2\ln|\langle e^{2i\theta}\rangle|}$$

This is the circular standard deviation of angle mod π (dipole has a 180° symmetry).

- Small σ_θ → stable direction → systematic template mis-registration.
- Large σ_θ → random direction → noise artefact (no coherent template shift).

A per-band scatter of angle vs MJD is also shown.

In [ ]:
def circular_std_deg(angles_deg: np.ndarray) -> float:
    """
    Circular standard deviation of an array of angles (degrees).
    Dipole has π-symmetry: fold to [0, 180) before computing.
    """
    a = np.deg2rad(angles_deg % 180)
    R = np.abs(np.mean(np.exp(2j * a)))
    return float(np.rad2deg(np.sqrt(-2 * np.log(R + 1e-12))) / 2)


angle_rows = []
for _, srow in top_sel.iterrows():
    oid = str(srow["diaObjectId"])
    df = src_cache.get(oid, pd.DataFrame())
    if df.empty or "r:isDipole" not in df.columns or "r:dipoleAngle" not in df.columns:
        continue
    df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
    if df_dip.empty:
        continue
    angles = pd.to_numeric(df_dip["r:dipoleAngle"], errors="coerce").dropna().values
    if len(angles) < 2:
        continue
    row = {
        "diaObjectId": oid,
        "n_dipoles": srow["n_dipoles"],
        "field": srow["field"],
        "angle_mean_deg": float(
            np.rad2deg(np.angle(np.mean(np.exp(2j * np.deg2rad(angles % 180)))) / 2) % 180
        ),
        "angle_circ_std_deg": circular_std_deg(angles),
        "length_median_arcsec": float(
            pd.to_numeric(df_dip.get("r:dipoleLength", pd.Series(dtype=float)), errors="coerce").median()
        ),
    }
    # Per-band circular std
    if "r:band" in df_dip.columns:
        for band in BAND_ORDER:
            ang_b = (
                pd.to_numeric(
                    df_dip[df_dip["r:band"] == band].get("r:dipoleAngle", pd.Series(dtype=float)),
                    errors="coerce",
                )
                .dropna()
                .values
            )
            row[f"angle_cstd_{band}"] = circular_std_deg(ang_b) if len(ang_b) >= 2 else np.nan
    angle_rows.append(row)

df_angles = pd.DataFrame(angle_rows).sort_values("angle_circ_std_deg").reset_index(drop=True)
print("Angular stability of dipole direction (sorted by circular std ascending):")
display(df_angles)
df_angles.to_csv(os.path.join(DIR_DATA, "dipole_angle_stability.csv"), index=False)

In [ ]:
# ── Rose histogram of dipole angles for all selected objects combined ─────────
all_angles = []
all_bands = []
for _, srow in top_sel.iterrows():
    oid = str(srow["diaObjectId"])
    df = src_cache.get(oid, pd.DataFrame())
    if df.empty or "r:isDipole" not in df.columns or "r:dipoleAngle" not in df.columns:
        continue
    df_dip = df[df["r:isDipole"].fillna(False).astype(bool)]
    if df_dip.empty:
        continue
    ang = pd.to_numeric(df_dip["r:dipoleAngle"], errors="coerce").dropna().values
    bnd = df_dip["r:band"].values[: len(ang)] if "r:band" in df_dip.columns else ["?"] * len(ang)
    all_angles.extend(ang % 180)  # fold to [0, 180) — dipole symmetry
    all_bands.extend(bnd)

if all_angles:
    fig, ax = plt.subplots(figsize=(7, 4))
    n_bins_angle = 36  # 5° bins
    bins_angle = np.linspace(0, 180, n_bins_angle + 1)
    bottom_bar = np.zeros(n_bins_angle)

    for band in BAND_ORDER:
        ang_b = np.array([a for a, b in zip(all_angles, all_bands) if b == band])
        if len(ang_b) == 0:
            continue
        counts_b, _ = np.histogram(ang_b, bins=bins_angle)
        ax.bar(
            (bins_angle[:-1] + bins_angle[1:]) / 2,
            counts_b,
            bottom=bottom_bar,
            width=180 / n_bins_angle,
            color=BAND_COLORS[band],
            edgecolor="white",
            lw=0.3,
            label=f"band {band}",
        )
        bottom_bar += counts_b

    ax.set_xlabel("Dipole angle mod 180° (degrees)")
    ax.set_ylabel("N detections")
    ax.set_xlim(0, 180)
    ax.set_title(
        f"Distribution of dipole angles — top {TOP_N_OBJECTS} objects\n"
        f"(folded mod 180° to account for dipole ±symmetry)"
    )
    ax.legend(loc="upper right", fontsize=8, ncol=3)
    plt.tight_layout()
    savefig(f"dipole_angle_histogram_top{TOP_N_OBJECTS}")
    plt.show()
else:
    print("No dipole angles available for the rose histogram.")

## 11. Per-field summary

Stacked dipole histogram split by DDF, restricted to pre-selected objects.

In [ ]:
n_fields = len(DEEP_FIELDS)
ncols = min(3, n_fields)
nrows = int(np.ceil(n_fields / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

for idx, fld in enumerate(DEEP_FIELDS):
    ax = axes[idx // ncols][idx % ncols]
    sub = df_stats[(df_stats["field"] == fld) & (df_stats["n_dipoles"] > 0)]
    if sub.empty:
        ax.set_title(f"{fld} — no dipoles")
        continue
    sub = sub.sort_values("n_dipoles", ascending=False)
    N_F = min(40, len(sub))
    top_f = sub.head(N_F)
    x_pos = np.arange(N_F)
    bottom = np.zeros(N_F)
    for band in BAND_ORDER:
        col = f"n_dip_{band}"
        if col not in top_f.columns:
            continue
        vals = top_f[col].values.astype(float)
        ax.bar(
            x_pos,
            vals,
            bottom=bottom,
            color=BAND_COLORS[band],
            edgecolor="white",
            lw=0.2,
            label=band,
            width=0.85,
        )
        bottom += vals
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(o)[-6:] for o in top_f["diaObjectId"]], rotation=90, fontsize=5)
    ax.set_title(f"{fld}  ({len(sub)} with >0 dip, top {N_F})", fontsize=8)
    ax.set_ylabel("N dipoles")
    ax.legend(loc="upper right", fontsize=6, ncol=3)

for idx in range(n_fields, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle(
    f"Dipole count per diaObject — per DDF (stacked by band, nDiaSrc>={NDIASOURCES_MIN})", fontsize=11, y=1.01
)
plt.tight_layout()
savefig(f"dipole_per_ddf_min{NDIASOURCES_MIN}")
plt.show()

## 12. Save all diaSources to Parquet

One combined Parquet file with all downloaded diaSources for the pre-selected objects,
plus one per-object file in a sub-directory for downstream use.

In [ ]:
subdir = os.path.join(DIR_DATA, "src_per_object")
os.makedirs(subdir, exist_ok=True)

all_src_list = []
for oid, df in src_cache.items():
    if df.empty:
        continue
    tmp = df.copy()
    tmp["diaObjectId_ext"] = oid  # ensure the ID is always present
    tmp["field"] = presel[oid]["field"]
    all_src_list.append(tmp)
    # Per-object file
    tmp.to_parquet(os.path.join(subdir, f"{oid}_src.parquet"), index=False)

if all_src_list:
    df_all_src = pd.concat(all_src_list, ignore_index=True)
    out_path = os.path.join(DIR_DATA, "all_src_presel.parquet")
    df_all_src.to_parquet(out_path, index=False)
    print(f"Saved {len(df_all_src):,} rows → {out_path}")
    print(f"Per-object files: {subdir}/")
else:
    print("No diaSources to save.")

## 13. Discussion and next steps

### Correct strategy recap

The conesearch returns **one alert per diaObject** (the most recent detection).
The only reliable per-object metadata it carries is `r:nDiaSources`.
Counting dipoles therefore requires downloading the **full diaSources** via
`/api/v1/sources`, which was done here for objects with `nDiaSources >= NDIASOURCES_MIN`.

### Key diagnostics

| Plot | Question answered |
|------|-------------------|
| Stacked bar histogram (section 7) | Which well-observed objects concentrate the dipoles? Which bands? |
| Lorenz curve (section 7) | How skewed is the distribution? |
| Light curves (section 9) | Are dipoles clustered in time or spread across the survey? |
| Dipole angle distribution (section 10) | Is the dipole direction stable (template offset) or random (noise)? |

### Suggested follow-up notebooks

- **`04_dipole_cutouts.ipynb`**: fetch science/template/difference image triplets
  from Fink (`/api/v1/cutouts`) for the dipole-flagged visits.
- **`05_dipole_butler.ipynb`**: use `r:visit` + `r:detector` from the diaSources
  to locate the corresponding frames in the USDF Butler and check the WCS registration quality.
- **`06_dipole_seeing.ipynb`**: join with `consDb` on `visitId` to correlate
  the per-visit dipole rate with observing conditions (seeing FWHM, airmass, sky brightness).
